In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import scipy.signal as signal
from scipy.fftpack import fft
from scipy.stats import kurtosis, entropy
from scipy.signal import get_window
import os


In [4]:
# Set sampling parameters
sampling_rate = 50000
window_size = int(sampling_rate / 50)
overlap = window_size // 2
stft_window_size = window_size // 2
channels = ['Channel1 [g]', 'Channel2 [g]', 'Channel3 [g]']

# Sample Y_true values (replace with your actual values)
y_true = [2.024, 1.983, 1.945, 2.384, 2.439, 2.467, 2.752, 2.812, 2.795, 2.185,
          2.024, 1.931, 2.405, 2.408, 2.481, 2.781, 2.748, 2.745, 2.085, 2.002,
          2.023, 2.441, 2.478, 2.440, 2.776, 2.693, 2.757]

file_paths = [f"part {i}.xlsx" for i in range(1, 28)]

In [14]:
def calculate_time_domain_features(signal):
    rms = np.sqrt(np.mean(signal**2))
    peak = np.max(np.abs(signal))
    crest_factor = peak / rms
    signal_kurtosis = kurtosis(signal)
    return rms, peak, crest_factor, signal_kurtosis
    
def calculate_frequency_features(signal_data):
    num_windows = (len(signal_data) - window_size) // window_size + 1
    psd_vals, dominant_freq_vals, total_energy_vals, spectral_entropy_vals = [], [], [], []
    for i in range(num_windows):
        start = i * window_size
        end = start + window_size
        if end > len(signal_data): break
        segment = signal_data[start:end]
        fft_transformed = np.abs(fft(segment))
        freqs = np.fft.fftfreq(len(segment), d=1/sampling_rate)
        psd_vals.append(np.mean(fft_transformed**2))
        dominant_freq_vals.append(freqs[np.argmax(fft_transformed)])
        total_energy_vals.append(np.sum(fft_transformed**2))
        psd_norm = fft_transformed / np.sum(fft_transformed)
        spectral_entropy_vals.append(entropy(np.abs(psd_norm)))
    return np.mean(psd_vals), np.mean(dominant_freq_vals), np.mean(total_energy_vals), np.mean(spectral_entropy_vals)

def compute_stft_features(signal_data, fs, window_size):
    frequencies, times, Zxx = signal.stft(signal_data, fs=fs, nperseg=window_size)
    magnitude = np.abs(Zxx) + 1e-10
    spectral_centroid = np.sum(frequencies[:, np.newaxis] * magnitude, axis=0) / np.sum(magnitude, axis=0)
    mean_centroid = np.nanmean(spectral_centroid)
    spectral_bandwidth = np.sqrt(np.sum(((frequencies[:, np.newaxis] - spectral_centroid) ** 2) * magnitude, axis=0) / np.sum(magnitude, axis=0))
    mean_bandwidth = np.nanmean(spectral_bandwidth)
    spectral_entropy = entropy(magnitude, axis=0)
    mean_entropy = np.nanmean(spectral_entropy)
    spectral_kurtosis = kurtosis(magnitude, axis=0, fisher=False)
    mean_kurtosis = np.nanmean(spectral_kurtosis)
    return mean_centroid, mean_bandwidth, mean_entropy, mean_kurtosis

In [15]:

# Load the data
data_frames = [pd.read_excel(file) for file in file_paths]

# Extract features
all_results = []
for part_idx, df in enumerate(data_frames):
    part_features = [f"Part {part_idx + 1}", part_idx + 1]
    for ch in channels:
        signal_data = df[ch].dropna().values
        time_feats = calculate_time_features(signal_data)
        freq_feats = calculate_frequency_features(signal_data)
        stft_feats = compute_stft_features(signal_data, sampling_rate, stft_window_size)
        part_features.extend([*time_feats, *freq_feats, *stft_feats])
    part_features.append(y_true[part_idx])
    all_results.append(part_features)

C:\Users\91834\AppData\Local\Temp\ipykernel_39924\1290468472.py:34: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  spectral_kurtosis = kurtosis(magnitude, axis=0, fisher=False)


In [19]:
# Define headers
column_headers = ["PART NO.", "PART NO. NUM"]
for ch in range(1, 4):  
    column_headers.extend([
        f"Channel {ch} RMS", f"Channel {ch} PEAK", f"Channel {ch} CREST FACTOR", f"Channel {ch} KURTOSIS",
        f"Channel {ch} PSD", f"Channel {ch} Dominant Frequency", f"Channel {ch} Total Energy", f"Channel {ch} Spectral Entropy",
        f"Channel {ch} STFT Centroid", f"Channel {ch} STFT Bandwidth", f"Channel {ch} STFT Entropy", f"Channel {ch} STFT Kurtosis"
    ])
column_headers.append("Y_true")

# Build DataFrame
features_df = pd.DataFrame(all_results, columns=column_headers)
features_df.sort_values(by="PART NO. NUM", inplace=True)
features_df.drop(columns=["PART NO. NUM"], inplace=True)
features_df.reset_index(drop=True, inplace=True)

# Step 1: Calculate R² for all 36 features
X = features_df.drop(columns=["PART NO.", "Y_true"])
Y = features_df["Y_true"]

# Convert to numeric to avoid dtype issues
X = X.apply(pd.to_numeric, errors='coerce')
Y = pd.to_numeric(Y, errors='coerce')

# Drop NaNs
valid_indices = Y.dropna().index
X = X.loc[valid_indices]
Y = Y.loc[valid_indices]

# Compute R² scores
r2_scores = {}
for col in X.columns:
    try:
        model = sm.OLS(Y, sm.add_constant(X[[col]])).fit()
        r2_scores[col] = model.rsquared
    except:
        r2_scores[col] = np.nan

# Step 2: Add R² as a row to the DataFrame
r2_row = {col: r2_scores.get(col, np.nan) for col in features_df.columns}
r2_row["PART NO."] = "R² Values"
features_df_with_r2 = pd.concat([features_df, pd.DataFrame([r2_row])], ignore_index=True)

# Step 3: Define feature groups
feature_groups = [
    "RMS", "PEAK", "CREST FACTOR", "KURTOSIS",
    "PSD", "Dominant Frequency", "Total Energy", "Spectral Entropy",
    "STFT Centroid", "STFT Bandwidth", "STFT Entropy", "STFT Kurtosis"
]

# Step 4: Select top feature in each group based on R²
selected_features = []
for group in feature_groups:
    group_features = [col for col in X.columns if group in col]
    if group_features:
        top_feature = max(group_features, key=lambda col: r2_scores.get(col, -np.inf))
        selected_features.append(top_feature)

# Step 5: Create top 12 DataFrame (excluding R² row)
top12_df = features_df_with_r2[features_df_with_r2["PART NO."] != "R² Values"]
top12_df = top12_df[["PART NO."] + selected_features + ["Y_true"]]

# Step 6: Save both sheets
#with pd.ExcelWriter("Vibration_Top12_Final.xlsx", engine="xlsxwriter") as writer:
   # features_df_with_r2.to_excel(writer, sheet_name="Combined_36_Features", index=False)
   # top12_df.to_excel(writer, sheet_name="Top_12_By_Group", index=False)

#print("✅ Saved with R² row and group-based top 12 feature selection.")



In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from itertools import combinations

# Custom accuracy function
def custom_accuracy(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(1 - np.abs(y_true - y_pred) / np.abs(y_true))

# Generate all 4-feature combinations
all_combinations = list(combinations(selected_features, 4))

# Define full input/output
X_full = top12_df[selected_features]
Y_true = top12_df["Y_true"]

# Initialize tracking variables
best_combination = None
best_accuracy = 0
best_r2 = 0
best_model = None
best_y_pred = None
results = []

# Loop through each 4-feature combination
for combo in all_combinations:
    X = X_full[list(combo)]
    model = LinearRegression().fit(X, Y_true)
    y_pred = model.predict(X)
    acc = custom_accuracy(Y_true, y_pred)
    r2 = r2_score(Y_true, y_pred)

    # Store results for later use if needed
    results.append({
        "Combination": combo,
        "Accuracy": acc,
        "R² Score": r2
    })

    # Track the best combination
    if acc > best_accuracy:
        best_combination = combo
        best_accuracy = acc
        best_r2 = r2
        best_model = model
        best_y_pred = y_pred

# Output best results
print("✅ Best 4-feature combination:", best_combination)
print(f"🎯 Best Custom Accuracy: {best_accuracy * 100:.2f}%")
print(f"📊 Best R² Score: {best_r2:.4f}")


✅ Best 4-feature combination: ('Channel 2 PEAK', 'Channel 3 CREST FACTOR', 'Channel 2 Spectral Entropy', 'Channel 2 STFT Bandwidth')
🎯 Best Custom Accuracy: 92.31%
📊 Best R² Score: 0.4824


In [21]:
# Create results DataFrame
results_df = pd.DataFrame(results)

# Create DataFrame with predictions for best combination
best_results_df = X_full[list(best_combination)].copy()
best_results_df["Y_true"] = Y_true
best_results_df["Y_pred"] = best_y_pred

# Save everything in one file
with pd.ExcelWriter("Vibration_Analysis_Model_Results.xlsx", engine="xlsxwriter") as writer:
    features_df_with_r2.to_excel(writer, sheet_name="Combined_36_Features", index=False)
    top12_df.to_excel(writer, sheet_name="Top_12_By_Group", index=False)
    results_df.to_excel(writer, sheet_name="All_4_Feature_Combos", index=False)
    best_results_df.to_excel(writer, sheet_name="Best_4_Feature_Result", index=False)

print("✅ All sheets saved in 'Vibration_Analysis_Model_Results.xlsx'")


✅ All sheets saved in 'Vibration_Analysis_Model_Results.xlsx'


In [27]:
y_true = [2.748, 2.059, 2.699, 2.037, 1.997, 2.658, 2.079, 2.059,
          2.039, 2.672, 2.101, 2.075, 2.569, 2.571, 2.573, 2.682]
# Sampling parameters
sampling_rate = 50000
window_size = int(sampling_rate / 50)
overlap = window_size // 2
stft_window_size = window_size // 2
channels = ['Channel1 [g]', 'Channel2 [g]', 'Channel3 [g]']

# Feature functions
def calculate_time_domain_features(signal):
    rms = np.sqrt(np.mean(signal**2))
    peak = np.max(np.abs(signal))
    crest_factor = peak / rms
    signal_kurtosis = kurtosis(signal)
    return rms, peak, crest_factor, signal_kurtosis

def calculate_frequency_features(signal_data):
    num_windows = (len(signal_data) - window_size) // window_size + 1
    psd_vals, dominant_freq_vals, total_energy_vals, spectral_entropy_vals = [], [], [], []
    for i in range(num_windows):
        start = i * window_size
        end = start + window_size
        if end > len(signal_data): break
        segment = signal_data[start:end]
        fft_transformed = np.abs(fft(segment))
        freqs = np.fft.fftfreq(len(segment), d=1/sampling_rate)
        psd_vals.append(np.mean(fft_transformed**2))
        dominant_freq_vals.append(freqs[np.argmax(fft_transformed)])
        total_energy_vals.append(np.sum(fft_transformed**2))
        psd_norm = fft_transformed / np.sum(fft_transformed)
        spectral_entropy_vals.append(entropy(np.abs(psd_norm)))
    return np.mean(psd_vals), np.mean(dominant_freq_vals), np.mean(total_energy_vals), np.mean(spectral_entropy_vals)

def compute_stft_features(signal_data, fs, window_size):
    frequencies, times, Zxx = signal.stft(signal_data, fs=fs, nperseg=window_size)
    magnitude = np.abs(Zxx) + 1e-10
    spectral_centroid = np.sum(frequencies[:, np.newaxis] * magnitude, axis=0) / np.sum(magnitude, axis=0)
    mean_centroid = np.nanmean(spectral_centroid)
    spectral_bandwidth = np.sqrt(np.sum(((frequencies[:, np.newaxis] - spectral_centroid) ** 2) * magnitude, axis=0) / np.sum(magnitude, axis=0))
    mean_bandwidth = np.nanmean(spectral_bandwidth)
    spectral_entropy = entropy(magnitude, axis=0)
    mean_entropy = np.nanmean(spectral_entropy)
    spectral_kurtosis = kurtosis(magnitude, axis=0, fisher=False)
    mean_kurtosis = np.nanmean(spectral_kurtosis)
    return mean_centroid, mean_bandwidth, mean_entropy, mean_kurtosis

# File paths for test parts (Part 28 to Part 43)
file_paths = [f"part {i}.xlsx" for i in range(28, 44)]

# Column headers
headers = ["PART NO.", "PART NO. NUM"]
for ch in range(1, 4):
    headers.extend([
        f"Channel {ch} RMS", f"Channel {ch} PEAK", f"Channel {ch} CREST FACTOR", f"Channel {ch} KURTOSIS",
        f"Channel {ch} PSD", f"Channel {ch} Dominant Frequency", f"Channel {ch} Total Energy", f"Channel {ch} Spectral Entropy",
        f"Channel {ch} STFT Centroid", f"Channel {ch} STFT Bandwidth",
        f"Channel {ch} STFT Entropy", f"Channel {ch} STFT Kurtosis"
    ])

# Extract 36 features for each part
all_test_results = []
for part_idx, file in enumerate(file_paths):
    df = pd.read_excel(file)
    row = [f"Part {part_idx + 28}", part_idx + 28]
    for ch in channels:
        signal_data = df[ch].dropna().values
        time_feats = calculate_time_domain_features(signal_data)
        freq_feats = calculate_frequency_features(signal_data)
        stft_feats = compute_stft_features(signal_data, sampling_rate, stft_window_size)
        row.extend([*time_feats, *freq_feats, *stft_feats])
    all_test_results.append(row)

# Create DataFrame
test_features_df = pd.DataFrame(all_test_results, columns=headers)
test_features_df.sort_values(by="PART NO. NUM", inplace=True)
test_features_df.drop(columns=["PART NO. NUM"], inplace=True)
test_features_df.reset_index(drop=True, inplace=True)


test_features_df["Y_true"] = y_true
testing_best4_df = test_features_df[["PART NO."] + list(best_combination) + ["Y_true"]].copy()

# Save both to Excel
with pd.ExcelWriter("Vibration_Test_Features_Final.xlsx", engine="xlsxwriter") as writer:
    test_features_df.to_excel(writer, sheet_name="Test_36_Features", index=False)
    testing_best4_df.to_excel(writer, sheet_name="Test_Best_4_Features", index=False)

print("✅ Test features saved with both 36 and best 4 in 'Vibration_Test_Features_Final.xlsx'")



✅ Test features saved with both 36 and best 4 in 'Vibration_Test_Features_Final.xlsx'
